# NB02 — Macro F_0.5 metric + stratified holdout

**Stage S1** of `Plan_ML_Challenge.md`. Two exit criteria:

1. `f_beta_per_entity` reproduces the problem statement's worked example
   (predicted `[S2-00047, S2-00193, S3-00812]`, gt `[S2-00047, S3-00812]` → **0.714**).
2. A stratified holdout of ~150k training S1 entities is persisted as parquet,
   stratified by country × cardinality bucket (D-03, D-08).

Copy-paste into SageMaker unchanged.


## 1. Config cell


In [1]:
# --- config cell: identical across notebooks ---
import os, sys, pathlib
ON_SAGEMAKER = pathlib.Path("/opt/ml").exists() or "SM_CURRENT_HOST" in os.environ
ROOT = pathlib.Path(os.environ.get(
    "BER_ROOT",
    "/home/ec2-user/SageMaker/ber" if ON_SAGEMAKER
    else "/Users/pocketfm/Desktop/amazon_ML/Working Folder"
))
DATA, ART, OUT = ROOT/"student_resource/dataset", ROOT/"artifacts", ROOT/"output"
sys.path.insert(0, str(ROOT/"src"))
for p in (ART, OUT): p.mkdir(parents=True, exist_ok=True)
N_JOBS = os.cpu_count() or 1
print(f"ON_SAGEMAKER={ON_SAGEMAKER}  ROOT={ROOT}  N_JOBS={N_JOBS}")


ON_SAGEMAKER=False  ROOT=/Users/pocketfm/Desktop/amazon_ML/Working Folder  N_JOBS=10


## 2. Unit test: reproduce the worked example (exit criterion #1)

The exact numbers from the problem statement plus a handful of edge cases the
scorer's singleton rule depends on.


In [2]:
from business_entity_resolution.metric import f_beta_per_entity, macro_f05

def close(a, b, tol=5e-4):
    return abs(a - b) < tol, f"{a} vs {b}"

# Worked example from the problem statement
pred = ["S2-00047", "S2-00193", "S3-00812"]
gt   = ["S2-00047", "S3-00812"]
s = f_beta_per_entity(pred, gt, beta=0.5)
ok, msg = close(s, 0.7143); assert ok, msg
print(f"  worked example                    : {s:.4f}  (expect 0.7143)  OK")

# Singleton edge cases (§ metric: gt empty → 1.0 iff pred empty; else 0.0)
assert f_beta_per_entity([], []) == 1.0
assert f_beta_per_entity(["a"], []) == 0.0
assert f_beta_per_entity([], ["a"]) == 0.0
print("  singleton edge cases              : OK")

# 4-truth, all 4 + 1 FP → 0.8333 (from Plan §1.5)
ok, _ = close(f_beta_per_entity(list("abcde"), list("abcd")), 0.8333); assert ok
# 2-truth, 2 confident correct                → 1.0 (P=1,R=1)
ok, _ = close(f_beta_per_entity(list("ab"), list("ab")), 1.0); assert ok
# 2-truth, 1 correct → P=1, R=0.5 → 0.8333
ok, _ = close(f_beta_per_entity(list("a"), list("ab")), 0.8333); assert ok
print("  Plan §1.5 threshold-behaviour table: OK")

# Macro across three entities with mixed situations
preds = {"S1-1": ["a","b"], "S1-2": [],  "S1-3": ["x"]}
truth = {"S1-1": ["a"],     "S1-2": [],  "S1-3": []}
# per-entity: P=0.5,R=1 → 5/9; empty=empty → 1.0; false-merge singleton → 0.0
expected = (5/9 + 1.0 + 0.0) / 3
ok, _ = close(macro_f05(preds, truth), expected, tol=1e-6); assert ok
print(f"  macro across mixed set            : {macro_f05(preds, truth):.4f}  OK")

print("\nS1 exit criterion #1 met: metric verified.")


  worked example                    : 0.7143  (expect 0.7143)  OK
  singleton edge cases              : OK
  Plan §1.5 threshold-behaviour table: OK
  macro across mixed set            : 0.5185  OK

S1 exit criterion #1 met: metric verified.


## 3. Load the training manifest

One row per training S1 entity with country, ground-truth match list, and
cardinality bucket. The bucket is `{0,1,2,3,4,5,6+}` — matches DECISIONS.md §A.2.


In [3]:
import time
from business_entity_resolution.config import detect_paths
from business_entity_resolution.splits import load_train_manifest

paths = detect_paths()
t0 = time.time()
manifest = load_train_manifest(
    paths.train / "train_source1.tsv",
    paths.train / "train_ground_truth.tsv",
)
print(f"loaded in {time.time()-t0:.1f}s  rows={len(manifest):,}")
manifest.head(3)


loaded in 6.0s  rows=2,206,821


,entity_id,country,gt_matches,n_true,cardinality_bucket
0,S1-925783039,US,"[S2-157377754, S2-517291332, S3-997698194, S3-...",4,4
1,S1-773889195,US,"[S2-970528089, S2-374107005, S3-189140669, S3-...",6,6
2,S1-377745466,US,"[S2-203037501, S3-70942743, S3-402918963, S3-8...",4,4


In [4]:
print("countries in training:")
print(manifest["country"].value_counts().to_string())
print("\ncardinality bucket distribution:")
print(manifest["cardinality_bucket"].value_counts(normalize=True).sort_index().round(4).to_string())
print(f"\nmean matches per S1: {manifest['n_true'].mean():.3f}  (expect 3.46)")


countries in training:
country
US       1323633
India     883188

cardinality bucket distribution:
cardinality_bucket
0    0.0558
1    0.0540
2    0.1700
3    0.2405
4    0.2194
5    0.1459
6    0.1143

mean matches per S1: 3.461  (expect 3.46)


## 4. Stratified holdout (exit criterion #2)

Target 150k held-out entities, stratified by (country × cardinality bucket) at a
single fraction (proportional-to-stratum-size). Preserves every stratum's marginal
share — balancing would distort the macro-F_0.5 estimate the holdout is here to
compute. Deterministic under `seed=0`.


In [5]:
from business_entity_resolution.splits import stratified_holdout, save_split

split = stratified_holdout(manifest, target_holdout=150_000, seed=0)
print(f"holdout = {len(split.holdout):,}   train = {len(split.train):,}")
assert len(split.holdout) + len(split.train) == len(manifest)
assert set(split.holdout.entity_id).isdisjoint(set(split.train.entity_id))
print("disjoint: OK")

# Marginal drift should be effectively zero
hold = split.holdout["cardinality_bucket"].value_counts(normalize=True).sort_index()
orig = manifest["cardinality_bucket"].value_counts(normalize=True).sort_index()
drift_bucket = (hold - orig).abs().max()
hc = split.holdout["country"].value_counts(normalize=True).sort_index()
oc = manifest["country"].value_counts(normalize=True).sort_index()
drift_country = (hc - oc).abs().max()
print(f"max drift (bucket) : {drift_bucket:.5f}")
print(f"max drift (country): {drift_country:.5f}")
assert drift_bucket < 5e-4 and drift_country < 5e-4

print("\nper-stratum split table:")
print(split.strata.to_string(index=False))


holdout = 149,999   train = 2,056,822


disjoint: OK
max drift (bucket) : 0.00000
max drift (country): 0.00001

per-stratum split table:
country  cardinality_bucket  n_total  n_holdout  frac
  India                   0    49351       3354 0.068
  India                   1    47468       3226 0.068
  India                   2   149927      10191 0.068
  India                   3   211965      14407 0.068
  India                   4   193669      13164 0.068
  India                   5   129339       8791 0.068
  India                   6   101469       6897 0.068
     US                   0    73896       5023 0.068
     US                   1    71689       4873 0.068
     US                   2   225285      15313 0.068
     US                   3   318876      21674 0.068
     US                   4   290446      19742 0.068
     US                   5   192618      13092 0.068
     US                   6   150823      10252 0.068


## 5. Persist to parquet (D-17)


In [6]:
files = save_split(split, paths.artifacts)
for k, v in files.items():
    p = pathlib.Path(v)
    print(f"  {k:8s} -> {p.name}  ({p.stat().st_size:>12,} bytes)")


  holdout  -> holdout_s1.parquet  (   6,952,630 bytes)
  train    -> train_s1.parquet  (  91,939,453 bytes)
  strata   -> split_strata.parquet  (       3,605 bytes)


## 6. Sanity: macro F_0.5 on the trivial "empty" baseline over the holdout

Every held-out entity gets an empty prediction. The score is the fraction of true
singletons in the holdout, which is A.2's 5.6% number carried onto our split — the
number the S0 submission would earn if uploaded.


In [7]:
empty_baseline = macro_f05(
    predictions={},
    truth=dict(zip(split.holdout["entity_id"], split.holdout["gt_matches"])),
)
singleton_frac = (split.holdout["n_true"] == 0).mean()
print(f"empty baseline on holdout   : {empty_baseline:.4f}")
print(f"singleton fraction (should equal above): {singleton_frac:.4f}")
assert abs(empty_baseline - singleton_frac) < 1e-9


empty baseline on holdout   : 0.0558
singleton fraction (should equal above): 0.0558


## 7. Log the run (D-19)


In [8]:
import json, time
row = {
    "ts": time.strftime("%Y-%m-%dT%H:%M:%S"),
    "stage": "S1",
    "notebook": "NB02_metric_and_splits",
    "config": {"target_holdout": 150_000, "seed": 0,
               "strata": ["country", "cardinality_bucket"]},
    "metrics": {
        "manifest_rows": int(len(manifest)),
        "holdout_rows": int(len(split.holdout)),
        "train_rows": int(len(split.train)),
        "empty_baseline_on_holdout": round(float(empty_baseline), 6),
        "mean_matches_per_s1": round(float(manifest["n_true"].mean()), 4),
        "max_bucket_drift": round(float(drift_bucket), 6),
        "max_country_drift": round(float(drift_country), 6),
    },
    "notes": "S1 exit criteria met: metric reproduces 0.714, stratified 150k holdout persisted.",
}
with open(ART/"runs.jsonl","a") as f:
    f.write(json.dumps(row) + "\n")
print(json.dumps(row, indent=2))


{
  "ts": "2026-09-26T16:30:23",
  "stage": "S1",
  "notebook": "NB02_metric_and_splits",
  "config": {
    "target_holdout": 150000,
    "seed": 0,
    "strata": [
      "country",
      "cardinality_bucket"
    ]
  },
  "metrics": {
    "manifest_rows": 2206821,
    "holdout_rows": 149999,
    "train_rows": 2056822,
    "empty_baseline_on_holdout": 0.055847,
    "mean_matches_per_s1": 3.4613,
    "max_bucket_drift": 4e-06,
    "max_country_drift": 6e-06
  },
  "notes": "S1 exit criteria met: metric reproduces 0.714, stratified 150k holdout persisted."
}


## 8. Next

S1 complete. Next stage is **S2 — rule baseline end-to-end + first leaderboard
upload** (`NB09_submission.ipynb`, filled by a first-cut rule matcher in
`src/business_entity_resolution/rule_baseline.py`). Exit criterion: valid submission
on the board with the local↔leaderboard offset recorded.

The holdout in `artifacts/holdout_s1.parquet` is the single source of truth every
later notebook should evaluate against — do not re-sample it.
